# 04: Instruction data and supervised fine-tuning

[Course index](../README_en.md) | [中文版本](../notebooks/04_sft.ipynb)

**Goal:** see how a tiny base model learns a narrow protocol without pretending to become a general assistant.


## 1. Why narrow tasks?

Pretraining learns a text distribution, not an arbitrary prompt protocol. A 17M model cannot acquire broad world knowledge from a small SFT set. MiniLLM therefore uses continuation, extractive QA, required-keyword stories, and sentence-count control—tasks grounded in its TinyStories domain.


## 2. Data format and leakage control

```text
Instruction: {instruction}
Input: {input}
Response:
```

The response is stored separately in JSONL. Splits are grouped by source story so related transformations never cross train/validation/test.


### Hands-on check

Read one real JSONL record and inspect prompt/response separation.


In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "model").exists():
    ROOT = ROOT.parents[2]
with (ROOT / "data/instruction_sft/train.jsonl").open(encoding="utf-8") as handle:
    sample = json.loads(next(handle))
print(sample["prompt"][:160])
print("response:", sample["response"][:80])
assert sample["prompt"].rstrip().endswith("Response:")
assert sample["response"]


## 3. Response-only loss

For response positions $R$:

$$\mathcal L_{SFT}=-rac1{|R|}\sum_{t\in R}\log p_	heta(y_t\mid x,y_{<t}).$$

Shift labels first, then set prompt targets to `-100`. This ordering avoids an off-by-one masking bug.


### Hands-on check

Use the production `SFTDataset` and inspect masked versus supervised labels.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "model").exists():
    ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))

from tokenizer.tokenizer_utils import MiniLLMTokenizer
from train.sft import SFTDataset

tokenizer = MiniLLMTokenizer(str(ROOT / "tokenizer/minillm_tokenizer.json"))
dataset = SFTDataset(
    str(ROOT / "data/instruction_sft/train.jsonl"), tokenizer, max_length=256
)
input_ids, labels = dataset[0]
mask = labels != -100
print("sequence tokens:", len(input_ids))
print("prompt targets ignored:", int((~mask).sum()))
print("response targets trained:", int(mask.sum()))
print("first response target:", tokenizer.decode([int(labels[mask][0])]))
assert mask.any() and (~mask).any()


## 4. Retaining base-model ability

MiniLLM mixes a small full-token language-model loss:

$$\mathcal L=\mathcal L_{SFT}+lpha\mathcal L_{LM},\qquadlpha=0.05.$$

This mitigates but cannot guarantee the absence of forgetting.


### Hands-on check

Combine toy SFT and LM losses with the project weight.


In [ ]:
import torch

sft_loss_demo = torch.tensor(1.20)
lm_loss_demo = torch.tensor(1.80)
combined = sft_loss_demo + 0.05 * lm_loss_demo
print("combined loss:", combined.item())
assert torch.isclose(combined, torch.tensor(1.29))


## 5. Formal training and fair control

```bash
python scripts/build_instruction_sft.py
python train/sft.py --config configs/train_sft.yaml
python train/sft.py --config configs/train_sft_continuation.yaml
```

Continuation-only SFT separates the effect of extra story training from the effect of task-structured supervision.


## 6. What to evaluate

Use QA exact match, exact sentence count, keyword coverage/all-pass, plus PPL and open-continuation repetition. Surface fluency is not equivalent to instruction success.

- [Previous](03_pretraining.ipynb) · [Index](../README_en.md) · [Next](05_alignment.ipynb)
- [`scripts/build_instruction_sft.py`](../../../scripts/build_instruction_sft.py)
- [`train/sft.py`](../../../train/sft.py)
